In [0]:
# dbutils.library.restartPython() is not supported on Serverless compute.
# sys.path setup in the next cell handles the src.config import.
#dbutils.library.restartPython()



# 03 · Qualidade de Dados na Camada Bronze
## 1. Objetivo e método

Esta etapa verifica a qualidade dos dados capturados na Bronze, antes de qualquer transformação. A ordem não é arbitrária: a verificação informa a transformação. Se a Silver fosse construída primeiro, a fonte de cada indicador seria escolhida sem sabero que há de errado nos dados — e a escolha óbvia seria, em vários casos, a errada.

### Estratégia: um pilar completo antes de cinco pela metade
O trabalho tem seis métricas alimentadas por sete fontes. Cobrir todas em profundidade simultaneamente produziria seis análises rasas e nenhum pipeline validado de ponta aponta. A escolha foi outra: **fechar o ciclo completo de um indicador — DEC e FEC — da Bronze até a Gold, e usar o resultado como molde para os demais**. 

Um pilar inteiro exercita todas as decisões de arquitetura em um caso real: composição de indicador a partir de parcelas, agregação ponderada, mudança de granularidade, construção de dimensão e fato,cálculo de evolução e ranqueamento. As métricas seguintes reaproveitam esse desenho em vez de reinventá-lo. 

O DEC e o FEC foram escolhidos como primeiro pilar por serem os indicadores mais complexos do conjunto: vêm em formato longo, exigem composição normativa a partir de onze parcelas, mudam de regra no meio da série histórica e precisam de agregação ponderada para subir de conjunto para distribuidora. O que funcionar aqui funciona para os outros.

### Consequência do prazo
A entrega tem data fixa. Fontes que não forem cobertas em profundidade têm a análise de qualidade registrada como pendência explícita, com o que falta verificar e por quê, na seção de fechamento deste notebook. A autoavaliação discute o que ficou de fora.

### Os cinco critérios

A verificação segue os critérios da especificação, aplicados a cada fonte:

| Critério | Pergunta |
|---|---|
| Completude | Existem valores nulos ou vazios? Em que proporção? |
| Consistência | Os valores seguem o padrão esperado, inclusive o do dicionário de dados? |
| Unicidade | Existem duplicatas onde não deveria haver? |
| Acurácia | Os valores fazem sentido no contexto e contra a regra de negócio? |
| Outliers | Existem valores extremos capazes de distorcer a análise? |

Cada fonte ocupa uma seção própria, com as mesmas sete subseções: estrutura, os cinco
critérios e uma síntese com os tratamentos definidos. Novas fontes entram como seções
irmãs, sem alterar a numeração das existentes.

## 2. Configuração

Constantes da janela de análise e o registro de achados, compartilhado por todas as seções. Cada verificação grava seu resultado em `FINDINGS`, e a síntese de cada fonte consolida o que foi encontrado.

In [0]:
import os
import sys

from pyspark.sql import functions as F
from pyspark.sql import Window

REPO_ROOT = os.path.dirname(os.getcwd())
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from src.config import CATALOG, SCHEMA_BRONZE

BRONZE = f"{CATALOG}.{SCHEMA_BRONZE}"

# Analysis window: full calendar years. Accumulating twelve months always in December
# avoids the distortion caused by mid-series changes in the set of consumer units,
# which distributors usually apply in January.
ANO_INICIAL = 2023
ANO_FINAL = 2025
ANOS_JANELA = list(range(ANO_INICIAL, ANO_FINAL + 1))

# Rounding tolerance accepted when comparing against ANEEL published aggregates
TOLERANCIA = 0.05

# Normative composition of DEC and FEC, PRODIST Module 8, Section 8.2.
# Two regimes: external parcels left the composition from 2022 onwards.
COMPOSICAO = {
    "ate_2021": ["IND", "IP", "XN", "XP"],
    "desde_2022": ["IND", "IP"],
}
ANO_MUDANCA_REGIME = 2022

# Own indicator: internal origin, unplanned, including ISE and Critical Day
PARCELAS_FI = ["IND", "INE", "INC"]

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA_BRONZE}")

print(f"Catalogo........: {BRONZE}")
print(f"Janela..........: {ANO_INICIAL} a {ANO_FINAL} (anos civis)")
print(f"Tolerancia......: {TOLERANCIA}")

In [0]:
# One row per quality check, consumed by the synthesis of each source section
FINDINGS = []

STATUS_OK = "OK"
STATUS_ATENCAO = "ATENCAO"
STATUS_PROBLEMA = "PROBLEMA"


def check(fonte, criterio, teste, status, detalhe, tratamento=""):
    """Register and print the result of one quality check.

    `fonte` is the Bronze table, `criterio` one of the five specification criteria,
    `teste` a short label, `status` one of the STATUS_* constants, `detalhe` what was
    measured and `tratamento` the decision taken when a problem was found.
    """
    FINDINGS.append({
        "fonte": fonte,
        "criterio": criterio,
        "teste": teste,
        "status": status,
        "detalhe": detalhe,
        "tratamento": tratamento,
    })
    print(f"[{status:<8}] {criterio:<12} {teste}")
    print(f"{'':>11}{detalhe}")
    if tratamento:
        print(f"{'':>11}Tratamento: {tratamento}")
    print()


def resumo_achados(fonte):
    """Print the consolidated findings of one source."""
    linhas = [f for f in FINDINGS if f["fonte"] == fonte]
    if not linhas:
        print("Nenhuma verificacao registrada.")
        return

    por_status = {}
    for f in linhas:
        por_status[f["status"]] = por_status.get(f["status"], 0) + 1

    print(f"{fonte}: {len(linhas)} verificacoes")
    for s in (STATUS_OK, STATUS_ATENCAO, STATUS_PROBLEMA):
        if s in por_status:
            print(f"  {s:<10} {por_status[s]}")
    print()
    return spark.createDataFrame(linhas)

## 3. Panorama das tabelas Bronze

Visão rasa de todas as tabelas carregadas: volume, número de colunas e tipos. Serve para dimensionar o conjunto e garantir que nenhuma fonte fique sem menção, mesmo aquelas cuja verificação aprofundada não couber no prazo.

In [0]:
tabelas = [r["tableName"] for r in spark.sql(f"SHOW TABLES IN {BRONZE}").collect()
           if not r["tableName"].startswith("_")]

panorama = []
for nome in sorted(tabelas):
    df = spark.table(f"{BRONZE}.{nome}")
    tipos = {}
    for _, dtype in df.dtypes:
        tipos[dtype] = tipos.get(dtype, 0) + 1
    panorama.append({
        "tabela": nome,
        "linhas": df.count(),
        "colunas": len(df.columns),
        "tipos": ", ".join(f"{k}:{v}" for k, v in sorted(tipos.items())),
    })

display(spark.createDataFrame(panorama))

## 4. Continuidade — `continuity_indicators`

Fonte dos indicadores DEC e FEC, base das métricas 3, 9 e 10. É a primeira a ser verificada em profundidade, conforme a estratégia da seção 1.

### 4.1 Estrutura e domínios

A analise da documentação disponibilizada pela ANEEL indica que a base vem em formato longo: uma linha por conjunto de unidades consumidoras, ano, mês e tipo de indicador. 
O valor está sempre em `VlrIndiceEnviado`, e o que ele significa depende de `SigIndicador`. 

Essa estrutura tem uma consequência prática: o mesmo campo carrega grandezas diferentes — horas de interrupção no DEC, número de interrupções no FEC e contagem de clientes no`NumCon`. Qualquer agregação sem filtrar por `SigIndicador` produz número sem sentido.

In [0]:
FONTE = "continuity_indicators"
cont = spark.table(f"{BRONZE}.{FONTE}")

print("=== schema ===")
for nome, tipo in cont.dtypes:
    print(f"  {nome:<26} {tipo}")

linhas = cont.count()
grao = cont.select("NumCNPJ", "IdeConjUndConsumidoras", "AnoIndice", "NumPeriodoIndice").distinct().count()
print(f"\nlinhas.....................: {linhas:,}")
print(f"grao conjunto x ano x mes..: {grao:,}")
print(f"indicadores por grao.......: {linhas / grao:.1f}")

# One label per CNPJ, reused by every check in this section so that the evidence shows
# who the distributor is. The code remains the identity; the label is only for reading.
siglas = (cont.select("NumCNPJ", F.trim("SigAgente").alias("SigAgente")).distinct()
              .groupBy("NumCNPJ").agg(F.max("SigAgente").alias("SigAgente")))

In [0]:
# Domain of SigIndicador, with volume and coverage per year
dominio = (cont
    .withColumn("SigIndicador", F.trim("SigIndicador"))
    .groupBy("SigIndicador")
    .agg(F.count("*").alias("linhas"),
         F.min("AnoIndice").alias("primeiro_ano"),
         F.max("AnoIndice").alias("ultimo_ano"),
         F.sum(F.when(F.col("VlrIndiceEnviado") > 0, 1).otherwise(0)).alias("linhas_com_valor"))
    .orderBy("SigIndicador"))

display(dominio)

In [0]:
# Period coverage: which months exist in each year
cobertura = (cont
    .groupBy("AnoIndice")
    .agg(F.min("NumPeriodoIndice").alias("primeiro_mes"),
         F.max("NumPeriodoIndice").alias("ultimo_mes"),
         F.countDistinct("NumPeriodoIndice").alias("meses"),
         F.countDistinct("IdeConjUndConsumidoras").alias("conjuntos"),
         F.countDistinct("NumCNPJ").alias("distribuidoras"))
    .orderBy("AnoIndice"))

display(cobertura)

anos = [r["AnoIndice"] for r in cobertura.collect()]
incompletos = [r["AnoIndice"] for r in cobertura.collect() if r["meses"] < 12]
check(FONTE, "Estrutura", "cobertura temporal",
      STATUS_OK if all(a in anos for a in ANOS_JANELA) else STATUS_PROBLEMA,
      f"Serie de {min(anos)} a {max(anos)}. Anos com menos de 12 meses: {incompletos or 'nenhum'}. "
      f"A janela {ANO_INICIAL}-{ANO_FINAL} esta integralmente coberta.",
      "Anos parciais ficam fora da janela por construcao, ao adotar anos civis completos.")

### 4.2 Completude:
Duas perguntas são respondidas aqui.

A primeira é a trivial: existem nulos nas colunas?

A segunda importa mais: cada parcela do DEC e do FEC está presente em todas as combinações de conjunto, ano e mês? Uma parcela ausente não gera nulo — ela simplesmente não vira linha, e a composição a trataria como zero, subestimando o indicador sem qualquer aviso.

In [0]:
# Null count per column
nulos = cont.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in cont.columns
]).collect()[0].asDict()

total = cont.count()
com_nulo = {k: v for k, v in nulos.items() if v > 0}
for coluna, qtd in sorted(nulos.items()):
    print(f"  {coluna:<26} {qtd:>10,}  ({qtd/total*100:.4f}%)")

check(FONTE, "Completude", "nulos por coluna",
      STATUS_OK if not com_nulo else STATUS_ATENCAO,
      f"Colunas com nulo: {com_nulo or 'nenhuma'}.",
      "" if not com_nulo else "Avaliar impacto por coluna antes da Silver.")

In [0]:
# Presence of each parcel across the full conjunto x ano x mes grid.
# A missing parcel does not produce a null: the row simply does not exist.
grade = cont.select("NumCNPJ", "IdeConjUndConsumidoras", "AnoIndice", "NumPeriodoIndice").distinct()
grade_por_ano = grade.groupBy("AnoIndice").count().withColumnRenamed("count", "grao_total")

presenca = (cont
    .withColumn("SigIndicador", F.trim("SigIndicador"))
    .groupBy("AnoIndice", "SigIndicador")
    .agg(F.count("*").alias("presentes"))
    .join(grade_por_ano, "AnoIndice")
    .withColumn("ausentes", F.col("grao_total") - F.col("presentes"))
    .withColumn("cobertura_pct", F.round(F.col("presentes") / F.col("grao_total") * 100, 2))
    .orderBy("SigIndicador", "AnoIndice"))

display(presenca.filter(F.col("AnoIndice").isin(ANOS_JANELA)))

In [0]:
# Parcels used by the composition must be complete inside the analysis window
usadas = [f"{ind}{p}" for ind in ("DEC", "FEC")
          for p in set(COMPOSICAO["desde_2022"] + PARCELAS_FI)]

faltas = (presenca
    .filter(F.col("AnoIndice").isin(ANOS_JANELA))
    .filter(F.col("SigIndicador").isin(usadas))
    .filter(F.col("ausentes") > 0)
    .orderBy(F.col("ausentes").desc()))

n_faltas = faltas.count()
# faltas has one row per parcel and year; the cells are what the fill would touch.
celulas_ausentes = (faltas.agg(F.coalesce(F.sum("ausentes"), F.lit(0)).alias("t"))
                          .collect()[0]["t"])

if n_faltas:
    display(faltas)

check(FONTE, "Completude", "presenca das parcelas usadas",
      STATUS_OK if n_faltas == 0 else STATUS_ATENCAO,
      f"Parcelas avaliadas: {sorted(usadas)}. Pares parcela-ano com ausencia na "
      f"janela: {n_faltas}, somando {celulas_ausentes:,} combinacoes conjunto-ano-mes.",
      "" if n_faltas == 0 else
      "Ausencia tratada como zero: a parcela INC so e informada quando ha Dia Critico, "
      "portanto a ausencia da linha equivale a ausencia de evento. Distribuidoras "
      "afetadas listadas na celula seguinte, para conferencia contra o publicado.")

In [0]:
# Which distributors, years and months are behind the missing parcels.
# Isolating them before the pivot fills with zero documents the fill instead of hiding it.
monitoradas = sorted(usadas)

grade_janela = (cont
    .filter(F.col("AnoIndice").isin(ANOS_JANELA))
    .select("NumCNPJ", "IdeConjUndConsumidoras", "AnoIndice", "NumPeriodoIndice")
    .distinct())

presentes = (cont
    .withColumn("SigIndicador", F.trim("SigIndicador"))
    .filter(F.col("AnoIndice").isin(ANOS_JANELA))
    .filter(F.col("SigIndicador").isin(monitoradas))
    .select("NumCNPJ", "IdeConjUndConsumidoras", "AnoIndice", "NumPeriodoIndice", "SigIndicador"))

# Expected grid: every grain combination times every monitored parcel.
esperado = grade_janela.crossJoin(
    spark.createDataFrame([(p,) for p in monitoradas], "SigIndicador string"))

ausencias = esperado.join(
    presentes,
    ["NumCNPJ", "IdeConjUndConsumidoras", "AnoIndice", "NumPeriodoIndice", "SigIndicador"],
    "left_anti")

if ausencias.limit(1).count():
    ausencias_dx = ausencias.join(siglas, "NumCNPJ")
    display(ausencias_dx
        .groupBy("SigAgente", "NumCNPJ", "SigIndicador", "AnoIndice")
        .agg(F.count("*").alias("celulas_ausentes"),
             F.countDistinct("IdeConjUndConsumidoras").alias("conjuntos_afetados"),
             F.sort_array(F.collect_set("NumPeriodoIndice")).alias("meses"))
        .orderBy(F.col("celulas_ausentes").desc()))
    # Kept for the reconciliation against the published indicator.
    dx_ausencia = [r["SigAgente"] for r in
                   ausencias_dx.select("SigAgente").distinct().orderBy("SigAgente").collect()]
    print(f"Distribuidoras com parcela ausente: {', '.join(dx_ausencia)}")
else:
    dx_ausencia = []
    print("Nenhuma parcela ausente entre as monitoradas.")

In [0]:
# NumCon is the weight of the weighted average; it must exist once per grain
peso = (cont
    .filter(F.trim("SigIndicador") == "NumCon")
    .select("NumCNPJ", "IdeConjUndConsumidoras", "AnoIndice", "NumPeriodoIndice", "VlrIndiceEnviado"))

peso_linhas = peso.count()
peso_grao = peso.select("NumCNPJ", "IdeConjUndConsumidoras",
                        "AnoIndice", "NumPeriodoIndice").distinct().count()
peso_zero = peso.filter(F.col("VlrIndiceEnviado") <= 0).count()
peso_nulo = peso.filter(F.col("VlrIndiceEnviado").isNull()).count()
grao_total = grade.count()

# Two opposite signals share the same arithmetic and must not be confused: grain without
# a weight is a gap; more rows than grain is duplication, which the uniqueness test owns.
ausente = grao_total - peso_grao
excedente = peso_linhas - peso_grao

print(f"grao total.................: {grao_total:,}")
print(f"grao com NumCon............: {peso_grao:,}")
print(f"linhas de NumCon...........: {peso_linhas:,}")
print(f"grao sem NumCon............: {ausente:,}")
print(f"linhas excedentes..........: {excedente:,}")
print(f"NumCon nulo................: {peso_nulo:,}")
print(f"NumCon zero ou negativo....: {peso_zero:,}")

if ausente or peso_zero or peso_nulo:
    status_peso = STATUS_PROBLEMA
    trat_peso = "Conjuntos sem peso valido nao podem ser agregados; definir tratamento."
elif excedente:
    status_peso = STATUS_ATENCAO
    trat_peso = ("Linhas excedentes sao duplicatas, nao ausencias. Resolvidas pela "
                 "deduplicacao da Silver e detalhadas no teste de unicidade.")
else:
    status_peso = STATUS_OK
    trat_peso = "NumCon da propria base e o peso da media ponderada; dispensa fonte externa."

check(FONTE, "Completude", "NumCon como peso", status_peso,
      f"NumCon cobre {peso_grao:,} de {grao_total:,} combinacoes do grao, com "
      f"{ausente:,} sem peso, {excedente:,} linhas excedentes, {peso_zero:,} zeros "
      f"e {peso_nulo:,} nulos.",
      trat_peso)

### 4.3 Consistência
Quatro verificações, da mais simples à que mais afeta o resultado. 

#### Comparar os tipos publicados com o dicionário de dados da ANEEL:
O dicionário especifica `NumCNPJ` como cadeia de 14 caracteres, `IdeConjUndConsumidoras` como cadeia de 5 e `AnoIndice` como cadeia de 4. O arquivo entrega os três como inteiro, o que destrói zeros à esquerda nos dois primeiros e quebra qualquer junção por esses campos. 

#### Estabilidade dos rótulos:
A ANEEL pode associar nomes distintos ao mesmo código de conjunto ao longo da série, e o mesmo ocorre com a sigla da distribuidora. Se isso se confirmar, nem `DscConjUndConsumidoras` nem `SigAgente` podem integrar a chave ou a dimensão — a identidade é o código, e o nome vem do de-para próprio. 

#### Verificar o domínio de `NumPeriodoIndice`:
Lista os valores distintos do campo e sua frequência, verificando que o domínio é exatamente 1 a 12, sem valores fora da faixa, lacunas ou strings.

#### Detectar reestruturação do conjunto de unidades consumidoras:
Distribuidoras reorganizam seus conjuntos, tipicamente em janeiro, transferindo subestações de um conjunto para outro. Quando isso ocorre, conjuntos novos entram sem histórico e o acumulado de doze meses fica artificialmente baixo. A verificação percorre toda a série a partir de 2022, comparando o conjunto de códigos de cada mês com o do mês anterior, e não apenas a virada de dezembro para janeiro.

A comparação é feita entre conjuntos de códigos, e não entre contagens, porque a transferência de subestações nem sempre altera a quantidade: a distribuidora ora troca o código, ora o mantém. Uma contagem simples não veria a segunda situação.

In [0]:
# Published types against the ANEEL data dictionary
esperado = {
    "NumCNPJ": ("string", 14),
    "IdeConjUndConsumidoras": ("string", 5),
    "AnoIndice": ("string", 4),
    "SigIndicador": ("string", 3),
    "SigAgente": ("string", 20),
    "DscConjUndConsumidoras": ("string", 255),
}
publicado = dict(cont.dtypes)

divergencias = []
for campo, (tipo_dic, tamanho) in esperado.items():
    tipo_pub = publicado.get(campo, "ausente")
    if not tipo_pub.startswith(tipo_dic):
        divergencias.append(f"{campo}: dicionario {tipo_dic}({tamanho}), publicado {tipo_pub}")

for d in divergencias:
    print(f"  {d}")

check(FONTE, "Consistencia", "tipos contra o dicionario",
      STATUS_OK if not divergencias else STATUS_PROBLEMA,
      f"{len(divergencias)} campos divergem do dicionario: "
      + "; ".join(divergencias) if divergencias else "Todos os campos conferem.",
      "Normalizar na Silver: CNPJ para 14 e conjunto para 5 caracteres, com zeros a esquerda."
      if divergencias else "")

In [0]:
# How many identifiers actually lose leading zeros when read as integer
ident = (cont
    .select(
        F.length(F.col("NumCNPJ").cast("string")).alias("len_cnpj"),
        F.length(F.col("IdeConjUndConsumidoras").cast("string")).alias("len_conj"))
    .groupBy("len_cnpj", "len_conj").count().orderBy("len_cnpj", "len_conj"))
display(ident)

cnpj_curto = cont.filter(F.length(F.col("NumCNPJ").cast("string")) < 14).count()
conj_curto = cont.filter(F.length(F.col("IdeConjUndConsumidoras").cast("string")) < 5).count()

check(FONTE, "Consistencia", "zeros a esquerda perdidos",
      STATUS_OK if (cnpj_curto + conj_curto) == 0 else STATUS_PROBLEMA,
      f"Linhas com CNPJ abaixo de 14 digitos: {cnpj_curto:,}. "
      f"Linhas com codigo de conjunto abaixo de 5 digitos: {conj_curto:,}.",
      "Aplicar lpad na Silver antes de qualquer juncao por esses campos."
      if (cnpj_curto + conj_curto) else "")

In [0]:
# Label stability: one code should map to one name
nomes_conjunto = (cont
    .groupBy("IdeConjUndConsumidoras")
    .agg(F.countDistinct(F.trim("DscConjUndConsumidoras")).alias("nomes"))
    .filter(F.col("nomes") > 1))

siglas_agente = (cont
    .groupBy("NumCNPJ")
    .agg(F.countDistinct(F.trim("SigAgente")).alias("siglas"))
    .filter(F.col("siglas") > 1))

n_conj = nomes_conjunto.count()
n_agt = siglas_agente.count()
total_conj = cont.select("IdeConjUndConsumidoras").distinct().count()
total_agt = cont.select("NumCNPJ").distinct().count()

print(f"conjuntos com mais de um nome....: {n_conj:,} de {total_conj:,}")
print(f"CNPJs com mais de uma sigla......: {n_agt:,} de {total_agt:,}")

if n_conj:
    display(cont.join(nomes_conjunto, "IdeConjUndConsumidoras")
                .select("IdeConjUndConsumidoras", "DscConjUndConsumidoras", "AnoIndice")
                .distinct().orderBy("IdeConjUndConsumidoras", "AnoIndice").limit(40))

check(FONTE, "Consistencia", "estabilidade de rotulos",
      STATUS_OK if (n_conj + n_agt) == 0 else STATUS_ATENCAO,
      f"{n_conj} de {total_conj} conjuntos tem mais de uma descricao; "
      f"{n_agt} de {total_agt} CNPJs tem mais de uma sigla.",
      "Identificacao por codigo. Nome e sigla vem do de-para proprio, fora da chave."
      if (n_conj + n_agt) else "")

In [0]:
# NumPeriodoIndice domain
periodos = sorted([r["NumPeriodoIndice"] for r in
                   cont.select("NumPeriodoIndice").distinct().collect()])
fora = [p for p in periodos if p < 1 or p > 12]

check(FONTE, "Consistencia", "dominio de NumPeriodoIndice",
      STATUS_OK if not fora else STATUS_PROBLEMA,
      f"Valores encontrados: {periodos}. Fora do intervalo 1 a 12: {fora or 'nenhum'}. "
      f"Confirma granularidade mensal unica, sem coexistencia de trimestral ou anual.",
      "" if not fora else "Investigar registros fora do intervalo antes de acumular.")

In [0]:
# Restructuring of consumer unit sets: full series from 2022 on.
# Comparing December against January alone would miss changes made in any other month.
# lag() looks at the previous EXISTING month, so a missing month shifts the comparison;
# read this together with the missing months check in 4.6.
from pyspark.sql import Window

ANO_INICIAL_REEST = 2022

conjuntos_mes = (cont
    .filter(F.col("AnoIndice") >= ANO_INICIAL_REEST)
    .withColumn("conj", F.lpad(F.col("IdeConjUndConsumidoras").cast("string"), 5, "0"))
    .groupBy("NumCNPJ", "AnoIndice", "NumPeriodoIndice")
    .agg(F.collect_set("conj").alias("conjuntos"))
    .withColumn("qtd_conjuntos", F.size("conjuntos")))

w_mes = Window.partitionBy("NumCNPJ").orderBy("AnoIndice", "NumPeriodoIndice")

reestruturacao = (conjuntos_mes
    .withColumn("conj_ant", F.lag("conjuntos").over(w_mes))
    .withColumn("qtd_ant", F.lag("qtd_conjuntos").over(w_mes))
    .filter(F.col("conj_ant").isNotNull())
    .withColumn("entrantes", F.size(F.array_except(F.col("conjuntos"), F.col("conj_ant"))))
    .withColumn("saintes", F.size(F.array_except(F.col("conj_ant"), F.col("conjuntos"))))
    .withColumn("delta", F.col("qtd_conjuntos") - F.col("qtd_ant"))
    .withColumn("mes_ref", F.concat_ws("-", F.col("AnoIndice"),
                                       F.lpad(F.col("NumPeriodoIndice"), 2, "0")))
    .filter((F.col("entrantes") > 0) | (F.col("saintes") > 0)))

n_reest = reestruturacao.count()
if n_reest:
    display(reestruturacao.join(siglas, "NumCNPJ")
            .select("SigAgente", "NumCNPJ", "mes_ref", "qtd_ant", "qtd_conjuntos",
                    "delta", "entrantes", "saintes")
            .orderBy(F.abs(F.col("delta")).desc()))

na_janela = reestruturacao.filter(F.col("AnoIndice").isin(ANOS_JANELA)).count()
fora_de_janeiro = reestruturacao.filter(F.col("NumPeriodoIndice") != 1).count()

check(FONTE, "Consistencia", "reestruturacao de conjuntos",
      STATUS_OK if na_janela == 0 else STATUS_ATENCAO,
      f"{n_reest} meses com entrada ou saida de conjuntos desde {ANO_INICIAL_REEST}; "
      f"{na_janela} dentro da janela {ANO_INICIAL}-{ANO_FINAL}; "
      f"{fora_de_janeiro} fora de janeiro.",
      "Acumular sempre em dezembro neutraliza o efeito dentro do ano civil. "
      "Distribuidoras que reestruturaram exigem cautela na comparacao entre blocos.")

### 4.4 Unicidade

O grão declarado é conjunto, ano, mês e indicador. Duplicata nesse grão significaria duplo envio da distribuidora ou falha de consolidação da ANEEL, e inflaria o indicador ao somar o mesmo valor duas vezes.

In [0]:
chave = ["NumCNPJ", "IdeConjUndConsumidoras", "AnoIndice", "NumPeriodoIndice", "SigIndicador"]

dup = (cont.withColumn("SigIndicador", F.trim("SigIndicador"))
           .groupBy(*chave).agg(F.count("*").alias("ocorrencias"),
                                F.countDistinct("VlrIndiceEnviado").alias("valores_distintos"))
           .filter(F.col("ocorrencias") > 1))

n_dup = dup.count()
if n_dup:
    display(dup.join(siglas, "NumCNPJ")
               .select("SigAgente", *chave, "ocorrencias", "valores_distintos")
               .orderBy(F.col("ocorrencias").desc()).limit(50))
    iguais = dup.filter(F.col("valores_distintos") == 1).count()
    detalhe = (f"{n_dup:,} chaves duplicadas, das quais {iguais:,} com valor identico "
               f"e {n_dup - iguais:,} com valores divergentes.")
    trat = ("Duplicata com valor identico: deduplicar. "
            "Com valor divergente: investigar antes de escolher o registro valido.")
else:
    detalhe = "Nenhuma duplicata no grao conjunto x ano x mes x indicador."
    trat = ""

check(FONTE, "Unicidade", "duplicatas no grao declarado",
      STATUS_OK if n_dup == 0 else STATUS_PROBLEMA, detalhe, trat)

### 4.5 Acurácia:

A verificação central desta fonte. O PRODIST, Módulo 8, Seção 8.2, define como o DEC eo FEC se compõem a partir das parcelas, e a definição mudou no meio da série:
- até dez/2021:   DEC = DECIND + DECIP + DECXN + DECXP 
- desde jan/2022: DEC = DECIND + DECIP

O FEC segue as parcelas equivalentes. A verificação recompõe o indicador a partir das parcelas apropriadas e compara com o valor consolidado publicado, em cada regime. 

Isso testa duas coisas ao mesmo tempo: que o pipeline implementa a regra oficial corretamente e que o dado publicado adere à própria norma. 

Divergências aqui não são erro do pipeline. São erro da fonte e é por isso que os indicadores deste trabalho são compostos a partir das parcelas, nunca lidos dos valores já pre-calculados pela fonte.

In [0]:
# Wide layout: one row per conjunto x ano x mes, one column per indicator.
# sorted() keeps the column order stable across runs.
todos_ind = sorted(r["SigIndicador"] for r in
                   cont.select(F.trim("SigIndicador").alias("SigIndicador")).distinct().collect())

# Duplicated keys would be resolved silently by F.first inside the pivot. Removing them
# here is analysis only: Bronze keeps the source as published and the deduplication
# belongs to Silver. Section 4.4 already established that no duplicate carries a
# divergent value, so dropping by key is safe.
cont_unico = cont.withColumn("SigIndicador", F.trim("SigIndicador")).dropDuplicates(chave)
removidas = cont.count() - cont_unico.count()
print(f"linhas duplicadas removidas para o pivo: {removidas:,}")

largo = (cont_unico
    .groupBy("NumCNPJ", "IdeConjUndConsumidoras", "AnoIndice", "NumPeriodoIndice")
    .pivot("SigIndicador", todos_ind)
    .agg(F.first("VlrIndiceEnviado"))
    .na.fill(0.0))

# Serverless compute does not support cache()/persist(). Materialising the pivot as a
# managed Delta table runs it once and gives the later cells a stable snapshot to read.
TABELA_LARGO = f"{BRONZE}.continuity_wide_tmp"
(largo.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_LARGO))

largo = spark.table(TABELA_LARGO)

print(f"grao conjunto x ano x mes: {largo.count():,} linhas, {len(largo.columns)} colunas")

In [0]:
def testar_composicao(df, indicador, parcelas, rotulo):
    """Compare the indicator recomposed from parcels against the published aggregate."""
    colunas = [f"{indicador}{p}" for p in parcelas]
    soma = sum(F.col(c) for c in colunas)
    res = (df
        .withColumn("soma_parcelas", soma)
        .withColumn("diferenca", F.col("soma_parcelas") - F.col(indicador))
        .withColumn("adere", F.abs(F.col("diferenca")) <= TOLERANCIA))

    agg = res.groupBy("AnoIndice").agg(
        F.count("*").alias("linhas"),
        F.sum(F.col("adere").cast("int")).alias("aderentes"),
        F.max(F.abs(F.col("diferenca"))).alias("maior_desvio"))
    agg = (agg.withColumn("aderencia_pct", F.round(F.col("aderentes") / F.col("linhas") * 100, 2))
              .withColumn("regime", F.lit(rotulo))
              .withColumn("indicador", F.lit(indicador))
              .orderBy("AnoIndice"))
    return res, agg


resultados = []
for indicador in ("DEC", "FEC"):
    for rotulo, parcelas in COMPOSICAO.items():
        alvo = (F.col("AnoIndice") < ANO_MUDANCA_REGIME if rotulo == "ate_2021"
                else F.col("AnoIndice") >= ANO_MUDANCA_REGIME)
        _, agg = testar_composicao(largo.filter(alvo), indicador, parcelas, rotulo)
        resultados.append(agg)

aderencia = resultados[0]
for r in resultados[1:]:
    aderencia = aderencia.unionByName(r)

display(aderencia.select("indicador", "regime", "AnoIndice", "linhas",
                         "aderentes", "aderencia_pct", "maior_desvio")
                 .orderBy("indicador", "AnoIndice"))

In [0]:
# Detail of the rows that do not adhere, under the regime in force since 2022
divergentes = None
for indicador in ("DEC", "FEC"):
    res, _ = testar_composicao(
        largo.filter(F.col("AnoIndice") >= ANO_MUDANCA_REGIME),
        indicador, COMPOSICAO["desde_2022"], "desde_2022")
    d = (res.filter(~F.col("adere"))
            .withColumn("indicador", F.lit(indicador))
            .select("indicador", "NumCNPJ", "IdeConjUndConsumidoras",
                    "AnoIndice", "NumPeriodoIndice",
                    F.round("soma_parcelas", 4).alias("soma_parcelas"),
                    F.round(F.col(indicador), 4).alias("publicado"),
                    F.round("diferenca", 4).alias("diferenca")))
    divergentes = d if divergentes is None else divergentes.unionByName(d)

# Serverless compute does not support cache()/persist(). This set is read here and again
# by the concentration cell, so it is materialised once as a work table.
TABELA_DIVERGENTES = f"{BRONZE}.continuity_divergences_tmp"
(divergentes.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_DIVERGENTES))

divergentes = spark.table(TABELA_DIVERGENTES)
n_div = divergentes.count()

por_empresa = (divergentes.join(siglas, "NumCNPJ")
    .groupBy("SigAgente", "NumCNPJ", "AnoIndice", "NumPeriodoIndice", "indicador")
    .agg(F.count("*").alias("conjuntos"),
         F.round(F.min("diferenca"), 4).alias("dif_min"),
         F.round(F.max("diferenca"), 4).alias("dif_max"))
    .orderBy(F.col("conjuntos").desc()))

display(por_empresa)

check(FONTE, "Acuracia", "composicao normativa do DEC e do FEC",
      STATUS_OK if n_div == 0 else STATUS_PROBLEMA,
      f"Regime desde 2022 (IND + IP): {n_div:,} combinacoes conjunto-ano-mes em que a soma "
      f"das parcelas difere do consolidado publicado em mais de {TOLERANCIA}.",
      "Compor o indicador sempre a partir das parcelas. O consolidado da ANEEL serve "
      "apenas como conferencia, com tolerancia de arredondamento.")

In [0]:
# Does the divergence reach the universe of interest? Concentration matters more
# than volume: an isolated error is noise, a systematic one is a finding.
concentracao = (divergentes.join(siglas, "NumCNPJ")
    .groupBy("SigAgente")
    .agg(F.count("*").alias("linhas"),
         F.countDistinct("IdeConjUndConsumidoras").alias("conjuntos"),
         F.countDistinct(F.concat_ws("-", "AnoIndice", "NumPeriodoIndice")).alias("meses"),
         F.round(F.max(F.abs(F.col("diferenca"))), 4).alias("maior_desvio"))
    .orderBy(F.col("linhas").desc()))

display(concentracao)

### 4.6 Outliers

Três frentes. 

- Valores impossíveis:

DEC ou FEC negativo e peso não positivo indicam erro de envio. 

- Valores extremos:

Valores extremos podem existir de fato — um temporal severo produz DEC alto e legítimo — e por isso não são removidos, apenas dimensionados, para que se saiba quanto do resultado depende de poucos eventos.

- Meses faltantes dentro de um ano

Uma Distribuidora pode apresentar falha ou atraso no envio dos indicadores que distorceriam o acumulado de doze meses sem gerar nulo algum. Torna-se necessário, nos indicadores LTM, a certificação de que são compostos por 12 meses. A ausencia de algum mês na série historica automaticamente eliminaria essa distribuidora da análise.

In [0]:
# Impossible values
negativos = {}
for coluna in [f"{i}{p}" for i in ("DEC", "FEC") for p in PARCELAS_FI] + ["DEC", "FEC", "NumCon"]:
    if coluna in largo.columns:
        n = largo.filter(F.col(coluna) < 0).count()
        if n:
            negativos[coluna] = n

peso_invalido = largo.filter(F.col("NumCon") <= 0).count()

check(FONTE, "Outliers", "valores impossiveis",
      STATUS_OK if not negativos and peso_invalido == 0 else STATUS_PROBLEMA,
      f"Colunas com valor negativo: {negativos or 'nenhuma'}. "
      f"Linhas com NumCon menor ou igual a zero: {peso_invalido:,}.",
      "" if not negativos and peso_invalido == 0
      else "Registros com valor impossivel nao podem compor o indicador.")

In [0]:
# Distribution of the own indicator, DEC-FI, inside the analysis window
janela = largo.filter(F.col("AnoIndice").isin(ANOS_JANELA))

for indicador in ("DEC", "FEC"):
    colunas = [f"{indicador}{p}" for p in PARCELAS_FI]
    janela = janela.withColumn(f"{indicador}_FI", sum(F.col(c) for c in colunas))

dist = janela.select(
    F.round(F.mean("DEC_FI"), 4).alias("dec_fi_media"),
    F.round(F.expr("percentile_approx(DEC_FI, 0.5)"), 4).alias("dec_fi_mediana"),
    F.round(F.expr("percentile_approx(DEC_FI, 0.99)"), 4).alias("dec_fi_p99"),
    F.round(F.max("DEC_FI"), 4).alias("dec_fi_max"),
    F.round(F.mean("FEC_FI"), 4).alias("fec_fi_media"),
    F.round(F.expr("percentile_approx(FEC_FI, 0.5)"), 4).alias("fec_fi_mediana"),
    F.round(F.expr("percentile_approx(FEC_FI, 0.99)"), 4).alias("fec_fi_p99"),
    F.round(F.max("FEC_FI"), 4).alias("fec_fi_max"))
display(dist)

p99 = janela.selectExpr("percentile_approx(DEC_FI, 0.99) as p").collect()[0]["p"]
extremos = janela.filter(F.col("DEC_FI") > p99)
n_ext = extremos.count()
peso_ext = (extremos.agg(F.sum(F.col("DEC_FI") * F.col("NumCon"))).collect()[0][0] or 0)
peso_tot = (janela.agg(F.sum(F.col("DEC_FI") * F.col("NumCon"))).collect()[0][0] or 1)

check(FONTE, "Outliers", "concentracao nos extremos",
      STATUS_ATENCAO if peso_ext / peso_tot > 0.25 else STATUS_OK,
      f"O 1% de combinacoes com maior DEC-FI (acima de {p99:.2f} h) responde por "
      f"{peso_ext/peso_tot*100:.1f}% da duracao total ponderada da janela.",
      "Extremos sao mantidos: temporal severo produz DEC alto e legitimo. "
      "A concentracao e reportada para dimensionar a sensibilidade do ranking.")

In [0]:
# Missing months inside a year distort the twelve month accumulation
meses_por_conjunto = (largo
    .filter(F.col("AnoIndice").isin(ANOS_JANELA))
    .groupBy("NumCNPJ", "IdeConjUndConsumidoras", "AnoIndice")
    .agg(F.countDistinct("NumPeriodoIndice").alias("meses")))

incompletos = meses_por_conjunto.filter(F.col("meses") < 12)
n_inc = incompletos.count()
n_tot = meses_por_conjunto.count()

if n_inc:
    display(incompletos.join(siglas, "NumCNPJ")
        .groupBy("SigAgente", "AnoIndice")
        .agg(F.count("*").alias("conjuntos_incompletos"),
             F.min("meses").alias("menor_cobertura"))
        .orderBy(F.col("conjuntos_incompletos").desc()))

check(FONTE, "Outliers", "meses faltantes no ano civil",
      STATUS_OK if n_inc == 0 else STATUS_ATENCAO,
      f"{n_inc:,} de {n_tot:,} combinacoes conjunto-ano tem menos de 12 meses na janela.",
      "" if n_inc == 0 else
      "Conjunto com ano incompleto tem acumulado subestimado. Definir na Silver se e "
      "excluido do bloco ou anualizado.")

### 4.7 Síntese: problemas encontrados e tratamentos definidos

Consolidação dos achados desta fonte e das decisões que eles produzem para a Silver.

In [0]:
display(resumo_achados(FONTE))

# Work tables created by 4.5. Dropping them here keeps the catalogue clean; comment the
# lines out to inspect the pivot or the divergence set after the run.
spark.sql(f"DROP TABLE IF EXISTS {TABELA_LARGO}")
spark.sql(f"DROP TABLE IF EXISTS {TABELA_DIVERGENTES}")

As decisões que a Silver de continuidade herda desta seção:

| Achado | Tratamento |
|---|---|
| Identificadores publicados como inteiro | `NumCNPJ` para texto de 14 e `IdeConjUndConsumidoras` para texto de 5, com zeros à esquerda, antes de qualquer junção |
| Rótulos instáveis para o mesmo código | Identificação por código; nome e sigla vêm do de-para próprio e ficam fora da chave |
| Linhas idênticas publicadas em duplicidade | `dropDuplicates` sobre a chave completa na Silver; a Bronze preserva o publicado |
| Parcela ausente no grão | `INC` tratada como zero, por só ser informada quando há Dia Crítico; distribuidoras afetadas conferidas contra o indicador publicado |
| Consolidado da ANEEL divergente da norma | Indicador sempre composto a partir das parcelas; consolidado apenas como conferência |
| Composição com dois regimes | Regra de composição condicionada ao ano; na janela 2023-2025 vale apenas o regime vigente desde 2022 |
| Reestruturação de conjuntos | Acumulação de doze meses sempre fechando em dezembro, o que confina o efeito a uma virada de bloco |
| `NumCon` completo e positivo | Peso da média ponderada, sem necessidade de fonte externa |

## Pendências documentadas

Fontes ainda não verificadas em profundidade, com o motivo e o que falta fazer. A seção cresce à medida que as seções por fonte são escritas, e o que permanecer aqui na entrega é discutido na autoavaliação.

| Fonte | Situação | O que falta |
|---|---|---|
| `emergency_occurrences_v1` e `_v2` | Não verificada | Validar o de-para entre os dois leiautes, em especial a hipótese de `DscOcorrenciaAberta` ter virado a quádrupla de fato gerador |
| `complaints` | Não verificada | Conferir o esquema entre os quatro anos e o filtro de nível 1 |
| `indger_commercial_services` e `commercial_quality` | Não verificada | Comparar universos e decidir a fonte da métrica 2 |
| `indger_commercial` | Não verificada | Validar `QtdUCAtiva` como denominador |
| `voltage_conformity` | Não verificada | Confirmar se `VlrLimite` é medida ou limite, e o tamanho da amostra |
| `pdd_investment` | Achados preliminares registrados | Duplicatas, CNPJ com duas siglas e valores como texto com vírgula decimal |

## Autoavaliação desta etapa

A ser escrita ao final da execução, cobrindo o que a etapa entregou, o que mudou no caminho, o que ficou em aberto e o que eu faria diferente.